# Experiment 05 — Dataset Size Scaling (BCE Standard)

**Goal:** Investigate how VQC performance scales from 500 to 5000 samples using Binary Cross-Entropy loss.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import roc_auc_score
from utils.data_utils import load_higgs

def train_scaling(n_samples):
    X_train, X_val, X_test, y_train, y_val, y_test = load_higgs(
        path='../data/HIGGS.csv.gz', n_samples=n_samples, n_features=2, scale_range=(0, np.pi)
    )
    
    dev = qml.device('lightning.qubit', wires=2)
    
    @qml.qnode(dev, interface='autograd')
    def circuit(weights, x):
        for i in range(2): qml.RY(x[i], wires=i)
        for l in range(2):
            for q in range(2): qml.Rot(*weights[l, q], wires=q)
            qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 0])
        return qml.expval(qml.PauliZ(0))

    def vqc_prob(w, x, b):
        return (circuit(w, x) + b + 1.0) / 2.0

    def bce_loss(weights, bias, X, y):
        probs = pnp.array([vqc_prob(weights, x, bias) for x in X])
        probs = pnp.clip(probs, 1e-15, 1.0 - 1e-15)
        return -pnp.mean(y * pnp.log(probs) + (1 - y) * pnp.log(1 - probs))

    pnp.random.seed(42)
    w = pnp.array(pnp.random.uniform(0, 2*np.pi, (2, 2, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=0.05)
    
    for epoch in range(20):
        idx = np.random.permutation(len(X_train))
        for start in range(0, len(X_train), 32):
            Xb, yb = X_train[idx][start:start+32], y_train[idx][start:start+32].astype(float)
            w, b = opt.step(lambda w_, b_: bce_loss(w_, b_, Xb, yb), w, b)
            
    test_probs = np.array([float(vqc_prob(w, x, b)) for x in X_test])
    return roc_auc_score(y_test, test_probs)

for ns in [500, 1000, 2500, 5000]:
    auc = train_scaling(ns)
    print(f"Samples: {ns:5d} | Test AUC: {auc:.4f}")